# Task 2: Text Chunking, Embedding and Vector Indexing

## Objective

Convert complaint narratives into vector embeddings and store them in a searchable vector database for semantic retrieval.

In [ ]:
import pandas as pd

sample_df = pd.read_csv(
    "../data/processed/sample_complaints.csv"
)

sample_df.shape

(13919, 20)

In [ ]:
import langchain
print(langchain.__version__)

1.3.11


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [ ]:
sample_df.columns.tolist()

['Date received',
 'Product',
 'Sub-product',
 'Issue',
 'Sub-issue',
 'Consumer complaint narrative',
 'Company public response',
 'Company',
 'State',
 'ZIP code',
 'Tags',
 'Consumer consent provided?',
 'Submitted via',
 'Date sent to company',
 'Company response to consumer',
 'Timely response?',
 'Consumer disputed?',
 'Complaint ID',
 'narrative_length',
 'clean_narrative']

In [ ]:
product_mapping = {

    "Credit card": "Credit Card",

    "Credit card or prepaid card": "Credit Card",

    "Checking or savings account": "Savings Account",

    "Money transfer, virtual currency, or money service":
        "Money Transfer",

    "Money transfers":
        "Money Transfer",

    "Consumer Loan":
        "Personal Loan",

    "Payday loan, title loan, or personal loan":
        "Personal Loan",

    "Payday loan, title loan, personal loan, or advance loan":
        "Personal Loan"
}


sample_df["product_category"] = (
    sample_df["Product"]
    .map(product_mapping)
)

In [ ]:
sample_df["product_category"].value_counts()

product_category
Credit Card        5680
Savings Account    4210
Money Transfer     2961
Personal Loan      1068
Name: count, dtype: int64

In [ ]:
sample_df.to_csv(
    "../data/processed/sample_complaints.csv",
    index=False
)

In [ ]:
documents = []

for _, row in sample_df.iterrows():

    chunks = splitter.split_text(
        str(row["clean_narrative"])
    )

    for idx, chunk in enumerate(chunks):

        documents.append(
            {
                "complaint_id": row["Complaint ID"],
                "product_category": row["product_category"],
                "product": row["Product"],
                "issue": row["Issue"],
                "company": row["Company"],
                "chunk_index": idx,
                "text": chunk
            }
        )

In [ ]:
chunks_df = pd.DataFrame(documents)

chunks_df.shape

(40101, 7)

In [ ]:
chunks_df.columns.tolist()

['complaint_id',
 'product_category',
 'product',
 'issue',
 'company',
 'chunk_index',
 'text']

In [ ]:
chunks_df.to_csv(
    "../data/processed/chunks.csv",
    index=False
)

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
import os

os.listdir("../data/processed")

['chunks.csv', 'filtered_complaints.csv', 'sample_complaints.csv']

In [ ]:
import pandas as pd

filtered_df = pd.read_csv(
    "../data/processed/filtered_complaints.csv"
)

filtered_df.shape

(463933, 21)

In [ ]:
import os

os.listdir("../data/raw")

['complaints.csv']

In [ ]:
import os

for root, dirs, files in os.walk("../"):
    for file in files:
        if "embedding" in file.lower():
            print(os.path.join(root, file))

../.venv\Lib\site-packages\cffi\_embedding.h
../.venv\Lib\site-packages\chromadb\db\mixins\embeddings_queue.py
../.venv\Lib\site-packages\chromadb\db\mixins\__pycache__\embeddings_queue.cpython-311.pyc
../.venv\Lib\site-packages\chromadb\migrations\embeddings_queue\00001-embeddings.sqlite.sql
../.venv\Lib\site-packages\chromadb\migrations\embeddings_queue\00002-embeddings-queue-config.sqlite.sql
../.venv\Lib\site-packages\chromadb\migrations\metadb\00001-embedding-metadata.sqlite.sql
../.venv\Lib\site-packages\chromadb\migrations\metadb\00002-embedding-metadata.sqlite.sql
../.venv\Lib\site-packages\chromadb\test\ef\test_chroma_bm25_embedding_function.py
../.venv\Lib\site-packages\chromadb\test\ef\__pycache__\test_chroma_bm25_embedding_function.cpython-311.pyc
../.venv\Lib\site-packages\chromadb\test\property\test_embeddings.py
../.venv\Lib\site-packages\chromadb\test\property\__pycache__\test_embeddings.cpython-311.pyc
../.venv\Lib\site-packages\chromadb\test\utils\test_embedding_funct

In [ ]:
import pandas as pd

chunks_df = pd.read_csv("../data/processed/chunks.csv")

print(chunks_df.shape)

(40101, 7)


In [ ]:
import pandas as pd

embeddings_df = pd.read_parquet(
    "../data/processed/complaint_embeddings.parquet"
)

print(embeddings_df.shape)

(1375327, 4)


In [ ]:
embeddings_df.head()

,id,document,embedding,metadata
0,14069121_0,a card was opened under my name by a fraudster...,"[-0.04277738183736801, 0.025624370202422142, -...","{'chunk_index': 0, 'company': 'CITIBANK, N.A.'..."
1,14061897_0,i made the mistake of using my wellsfargo debi...,"[-0.05458317697048187, 0.10340359061956406, 0....","{'chunk_index': 0, 'company': 'WELLS FARGO & C..."
2,14061897_1,and got a letter stating my dispute was reject...,"[-0.03491289168596268, 0.059216588735580444, 0...","{'chunk_index': 1, 'company': 'WELLS FARGO & C..."
3,14047085_0,"dear cfpb, i have a secured credit card with c...","[-0.010181158781051636, 0.02354264445602894, -...","{'chunk_index': 0, 'company': 'CITIBANK, N.A.'..."
4,14047085_1,y confirmation whatsoever to report to the pol...,"[-0.017308838665485382, -0.007177562452852726,...","{'chunk_index': 1, 'company': 'CITIBANK, N.A.'..."


In [ ]:
embeddings_df.columns.tolist()

['id', 'document', 'embedding', 'metadata']

In [ ]:
embeddings_df["metadata"].iloc[0]

{'chunk_index': 0,
 'company': 'CITIBANK, N.A.',
 'complaint_id': '14069121',
 'date_received': '2025-06-13',
 'issue': 'Getting a credit card',
 'product': 'Credit card',
 'product_category': 'Credit Card',
 'state': 'TX',
 'sub_issue': 'Card opened without my consent or knowledge',
 'total_chunks': 1}

In [ ]:
type(embeddings_df["metadata"].iloc[0])

dict

In [ ]:
embeddings_df["metadata"].iloc[0].keys()

dict_keys(['chunk_index', 'company', 'complaint_id', 'date_received', 'issue', 'product', 'product_category', 'state', 'sub_issue', 'total_chunks'])

In [ ]:
len(embeddings_df["embedding"].iloc[0])

384

In [ ]:
import numpy as np

embedding_matrix = np.array(
    embeddings_df["embedding"].tolist(),
    dtype=np.float32
)

embedding_matrix.shape

(1375327, 384)

In [ ]:
question = "Why are customers unhappy with credit cards?"

query_embedding = model.encode(question)

print(query_embedding.shape)

(384,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
small_matrix = embedding_matrix[:50000]

small_df = embeddings_df.iloc[:50000]

In [ ]:
similarities = cosine_similarity(
    [query_embedding],
    small_matrix
)[0]

In [ ]:
import numpy as np

top_indices = np.argsort(similarities)[-5:][::-1]

top_indices

array([ 8231,  4173, 49351,  7664,   272])

In [ ]:
for idx in top_indices:

    print("="*80)

    print("Similarity:", similarities[idx])

    print()

    print(small_df.iloc[idx]["document"][:500])

    print()

    print(small_df.iloc[idx]["metadata"])

Similarity: 0.6402717

well so this will help a lot as ive noted to them. this feels retaliatory, especially following my apr request, and harmful to customers like me who are genuinely trying to manage debt responsibly during personal hardship. the balance on this card is way below the limit. i have also been paying more than the minimum every month. my credit score has only dropped around pointsnothing extreme and still considered a fair score and ive been doing my best to maintain my financial health despite a chal

{'chunk_index': 1, 'company': 'AMERICAN EXPRESS COMPANY', 'complaint_id': '13012072', 'date_received': '2025-04-16', 'issue': "Problem with a company's investigation into an existing problem", 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'CA', 'sub_issue': 'Difficulty submitting a dispute or getting information about a dispute over the phone', 'total_chunks': 4}
Similarity: 0.63569176

ding to their own information, i have good credit. when a loy

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_chunks(question, k=5):

    query_embedding = model.encode(question)

    similarities = cosine_similarity(
        [query_embedding],
        embedding_matrix
    )[0]

    top_indices = np.argsort(similarities)[-k:][::-1]

    results = []

    for idx in top_indices:

        results.append({
            "text": embeddings_df.iloc[idx]["document"],
            "metadata": embeddings_df.iloc[idx]["metadata"],
            "score": similarities[idx]
        })

    return results

In [ ]:
results = retrieve_chunks(
    "Why are customers unhappy with credit cards?"
)

len(results)

5

In [ ]:
for r in results:

    print("="*80)

    print("Score:", r["score"])

    print()

    print(r["text"][:500])

    print()

    print(r["metadata"])

Score: 0.713571

card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.

{'chunk_index': 2, 'company': 'BARCLAYS BANK DELAWARE', 'complaint_id': '3509835', 'date_received': '2020-01-27', 'issue': 'Other features, terms, or problems', 'product': 'Credit card or prepaid card', 'product_category': 'Credit Card', 'state': 'CA', 'sub_issue': 'Problem with customer service', 'total_chunks': 4}
Score: 0.71128213

creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my 

In [ ]:
def build_context(results):

    context = ""

    for i, r in enumerate(results, start=1):

        context += f"""
Chunk {i}

{r['text']}

"""

    return context

In [ ]:
context = build_context(results)

print(context[:1000])


Chunk 1

card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.


Chunk 2

creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these companies it like here is the card but don't use it!


Chunk 3

credit card companies think of their card holders. the indifferent response to a long-term card holder again came to me as a shock given how this particular company was receptive to me in the past when i called about othe

In [ ]:
PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust.

Use ONLY the complaint excerpts below.

Identify common themes, customer frustrations,
and recurring issues.

If the information is not available,
say that you do not have enough information.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
question = "Why are customers unhappy with credit cards?"

prompt = PROMPT_TEMPLATE.format(
    context=context,
    question=question
)

print(prompt[:3000])


You are a financial analyst assistant for CrediTrust.

Use ONLY the complaint excerpts below.

Identify common themes, customer frustrations,
and recurring issues.

If the information is not available,
say that you do not have enough information.

Context:

Chunk 1

card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.


Chunk 2

creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these companies it like here is t

In [ ]:
del generator

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=-1
)

print("Generator loaded")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

c:\Users\HP\rag-complaint-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

In [2]:
def ask_rag(question, k=5):

    # Retrieve relevant complaint chunks
    results = retrieve_chunks(question, k=k)

    # Build context from retrieved chunks
    context = build_context(results)

    # Create prompt
    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

    # Generate answer
    response = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )

    answer = response[0]["generated_text"]

    return answer, results

In [7]:
import pandas as pd
import numpy as np

In [8]:
embeddings_df = pd.read_parquet(
    "../data/processed/complaint_embeddings.parquet"
)

embeddings_df.head()

,id,document,embedding,metadata
0,14069121_0,a card was opened under my name by a fraudster...,"[-0.04277738183736801, 0.025624370202422142, -...","{'chunk_index': 0, 'company': 'CITIBANK, N.A.'..."
1,14061897_0,i made the mistake of using my wellsfargo debi...,"[-0.05458317697048187, 0.10340359061956406, 0....","{'chunk_index': 0, 'company': 'WELLS FARGO & C..."
2,14061897_1,and got a letter stating my dispute was reject...,"[-0.03491289168596268, 0.059216588735580444, 0...","{'chunk_index': 1, 'company': 'WELLS FARGO & C..."
3,14047085_0,"dear cfpb, i have a secured credit card with c...","[-0.010181158781051636, 0.02354264445602894, -...","{'chunk_index': 0, 'company': 'CITIBANK, N.A.'..."
4,14047085_1,y confirmation whatsoever to report to the pol...,"[-0.017308838665485382, -0.007177562452852726,...","{'chunk_index': 1, 'company': 'CITIBANK, N.A.'..."


In [9]:
embedding_matrix = np.vstack(
    embeddings_df["embedding"].values
)

print(embedding_matrix.shape)

(1375327, 384)


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\HP\rag-complaint-chatbot\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_chunks(question, k=5):

    query_embedding = model.encode(question)

    similarities = cosine_similarity(
        [query_embedding],
        embedding_matrix
    )[0]

    top_indices = np.argsort(similarities)[-k:][::-1]

    results = []

    for idx in top_indices:

        results.append({
            "text": embeddings_df.iloc[idx]["document"],
            "metadata": embeddings_df.iloc[idx]["metadata"],
            "score": similarities[idx]
        })

    return results

In [12]:
def build_context(results):

    context = ""

    for i, r in enumerate(results, start=1):

        context += f"""

Chunk {i}

{r['text']}

"""

    return context

In [13]:
PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust.

Use ONLY the complaint excerpts below.

Identify common themes, customer frustrations,
and recurring issues.

If the information is not available,
say that you do not have enough information.

Context:
{context}

Question:
{question}

Answer:
"""

In [14]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1
)

c:\Users\HP\rag-complaint-chatbot\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [15]:
def ask_rag(question, k=5):

    results = retrieve_chunks(question, k=k)

    context = build_context(results)

    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

    answer = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )[0]["generated_text"]

    return answer, results

In [16]:
answer, sources = ask_rag(
    "Why are customers unhappy with credit cards?"
)

print(answer)

They don't understand the features and benefits


# ==========================
# RAG Evaluation
# ==========================

In [17]:
# ==========================
# RAG Evaluation
# ==========================

In [18]:
questions = [

    "Why are customers unhappy with credit cards?",

    "What problems do customers report about personal loans?",

    "What are common complaints about savings accounts?",

    "Why do customers complain about money transfers?",

    "Which companies receive complaints about customer service?",

    "What fraud-related complaints are common?",

    "Why are customers disputing credit card charges?",

    "What issues are most common with interest rates?"

]

In [19]:
for q in questions:

    print("="*100)

    print("QUESTION:")

    print(q)

    print()

    answer, sources = ask_rag(q)

    print("ANSWER:")

    print(answer)

    print()

    print("TOP SOURCE:")

    print(sources[0]["metadata"])

    print()

QUESTION:
Why are customers unhappy with credit cards?

ANSWER:
They don't understand the features and benefits

TOP SOURCE:
{'chunk_index': 2, 'company': 'BARCLAYS BANK DELAWARE', 'complaint_id': '3509835', 'date_received': '2020-01-27', 'issue': 'Other features, terms, or problems', 'product': 'Credit card or prepaid card', 'product_category': 'Credit Card', 'state': 'CA', 'sub_issue': 'Problem with customer service', 'total_chunks': 4}

QUESTION:
What problems do customers report about personal loans?

ANSWER:
recurring issues

TOP SOURCE:
{'chunk_index': 2, 'company': 'BMO Bank, N.A.', 'complaint_id': '12783568', 'date_received': '2025-04-02', 'issue': 'Problem when making payments', 'product': 'Payday loan, title loan, personal loan, or advance loan', 'product_category': 'Personal Loan', 'state': 'NC', 'sub_issue': 'nan', 'total_chunks': 3}

QUESTION:
What are common complaints about savings accounts?



Token indices sequence length is longer than the specified maximum sequence length for this model (537 > 512). Running this sequence through the model will result in indexing errors


ANSWER:
Customer frustrations

TOP SOURCE:
{'chunk_index': 4, 'company': 'M&T BANK CORPORATION', 'complaint_id': '11374851', 'date_received': '2025-01-03', 'issue': 'Managing an account', 'product': 'Checking or savings account', 'product_category': 'Savings Account', 'state': 'NY', 'sub_issue': 'Deposits and withdrawals', 'total_chunks': 6}

QUESTION:
Why do customers complain about money transfers?

ANSWER:
They are being discriminatory because my friend is in

TOP SOURCE:
{'chunk_index': 4, 'company': 'WELLS FARGO & COMPANY', 'complaint_id': '2905131', 'date_received': '2018-05-13', 'issue': 'Fraud or scam', 'product': 'Money transfer, virtual currency, or money service', 'product_category': 'Money Transfer', 'state': 'CA', 'sub_issue': 'nan', 'total_chunks': 5}

QUESTION:
Which companies receive complaints about customer service?

ANSWER:
corporate

TOP SOURCE:
{'chunk_index': 6, 'company': 'U.S. BANCORP', 'complaint_id': '8149597', 'date_received': '2024-01-12', 'issue': 'Managing

In [20]:
evaluation = []

In [21]:
for q in questions:

    answer, sources = ask_rag(q)

    evaluation.append({

        "Question": q,

        "Generated Answer": answer,

        "Top Source": sources[0]["metadata"]["company"],

        "Score": "",

        "Comments": ""

    })

In [22]:
evaluation_df = pd.DataFrame(evaluation)

evaluation_df

,Question,Generated Answer,Top Source,Score,Comments
0,Why are customers unhappy with credit cards?,They don't understand the features and benefits,BARCLAYS BANK DELAWARE,,
1,What problems do customers report about person...,recurring issues,"BMO Bank, N.A.",,
2,What are common complaints about savings accou...,Customer frustrations,M&T BANK CORPORATION,,
3,Why do customers complain about money transfers?,They are being discriminatory because my frien...,WELLS FARGO & COMPANY,,
4,Which companies receive complaints about custo...,corporate,U.S. BANCORP,,
5,What fraud-related complaints are common?,"emotional coercion, soulmate scams, billing co...",Regional Management Corporation,,
6,Why are customers disputing credit card charges?,a consumer group that is usually in financial ...,"Bread Financial Holdings, Inc.",,
7,What issues are most common with interest rates?,"high interest rates, denial of credit",OneMain Finance Corporation,,


In [23]:
evaluation_df.to_csv(
    "../evaluation.csv",
    index=False
)